In [9]:
! pip install ollama

In [10]:
language_model = " hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF:latest"
embedding_model = " hf.co/CompendiumLabs/bge-base-en-v1.5-gguf:latest"

In [11]:
datasets = []
with open("cat-facts.txt", "r", encoding="utf-8") as f:
    datasets = f.readlines()
    print(f"Loaded {len(datasets)} facts")

Loaded 150 facts


In [16]:
VECTOR_DB = []

In [19]:
def add_chunk_to_vector_db(chunk):
    embeddings = ollama.embed(model=embedding_model, input=chunk)['embeddings'][0]
    VECTOR_DB.append((chunk, embeddings))

In [20]:
for i, chunk in enumerate(datasets):
    add_chunk_to_vector_db(chunk)
    print(f"Added chunk {i+1}/{len(datasets)} to vector database")

Added chunk 1/150 to vector database
Added chunk 2/150 to vector database
Added chunk 3/150 to vector database
Added chunk 4/150 to vector database
Added chunk 5/150 to vector database
Added chunk 6/150 to vector database
Added chunk 7/150 to vector database
Added chunk 8/150 to vector database
Added chunk 9/150 to vector database
Added chunk 10/150 to vector database
Added chunk 11/150 to vector database
Added chunk 12/150 to vector database
Added chunk 13/150 to vector database
Added chunk 14/150 to vector database
Added chunk 15/150 to vector database
Added chunk 16/150 to vector database
Added chunk 17/150 to vector database
Added chunk 18/150 to vector database
Added chunk 19/150 to vector database
Added chunk 20/150 to vector database
Added chunk 21/150 to vector database
Added chunk 22/150 to vector database
Added chunk 23/150 to vector database
Added chunk 24/150 to vector database
Added chunk 25/150 to vector database
Added chunk 26/150 to vector database
Added chunk 27/150 to

In [21]:
VECTOR_DB

[('On average, cats spend 2/3 of every day sleeping. That means a nine-year-old cat has been awake for only three years of its life.\n',
  [-0.035516445,
   -0.023937393,
   0.047546096,
   -0.079606496,
   0.036145028,
   -0.014062932,
   0.07793605,
   0.052710805,
   -0.014562385,
   -0.0027839248,
   -0.019629328,
   -0.008097693,
   -0.05905563,
   0.01173104,
   -0.046903722,
   0.04087853,
   0.087709926,
   0.010635704,
   -0.031251855,
   -0.029040515,
   0.004663734,
   0.02645779,
   -0.025628256,
   -0.0002798607,
   0.051095348,
   -0.0137546575,
   -0.0122575285,
   -0.0033963262,
   0.0028038868,
   -0.011705697,
   0.03738507,
   -0.031067425,
   -0.033931077,
   0.005881986,
   0.027011259,
   -0.035031196,
   -0.01343732,
   -0.032872148,
   0.014325312,
   0.033897642,
   -0.033218212,
   -0.013505012,
   -0.01889466,
   0.015251422,
   -0.029835785,
   -0.01749344,
   -0.001858668,
   -0.013608489,
   0.02690408,
   0.01695673,
   -0.032779153,
   0.0014614781,
   0

In [24]:
def cosine_similarity(vec1, vec2):
    dot_product = sum(a * b for a, b in zip(vec1, vec2))
    magnitude1 = sum(a ** 2 for a in vec1) ** 0.5
    magnitude2 = sum(b ** 2 for b in vec2) ** 0.5
    if magnitude1 == 0 or magnitude2 == 0:
        return 0.0
    return dot_product / (magnitude1 * magnitude2)

[-0.011643255,
 -0.030210348,
 0.065172024,
 -0.035883576,
 0.03756912,
 0.01502146,
 0.05030625,
 0.049945526,
 -0.0002914262,
 -0.038718957,
 -0.007150723,
 -0.012533859,
 -0.018993987,
 0.01627066,
 -0.027863631,
 0.055670414,
 0.044554453,
 0.018507661,
 -0.02413178,
 -0.02334759,
 -0.046617225,
 0.0006794038,
 -0.04651847,
 0.01864098,
 0.04779447,
 0.04358275,
 0.0059025185,
 -0.00308121,
 -0.034924872,
 -0.010934713,
 0.043452755,
 -0.02456626,
 -0.03964818,
 -0.0052923877,
 0.035632282,
 -0.04729529,
 -0.037764605,
 0.0039697927,
 0.0022765333,
 -0.0074639544,
 -0.014587592,
 0.014604044,
 0.02668809,
 -0.005008724,
 -0.030756257,
 0.00527672,
 0.0024181535,
 -0.031339444,
 0.0016018248,
 0.008627698,
 -0.04753837,
 0.019308029,
 -0.070645034,
 0.0046031675,
 -0.0053995918,
 0.005737714,
 0.07465062,
 -0.017793776,
 -0.0335974,
 0.01705154,
 0.029898025,
 -0.026463766,
 0.008924201,
 -0.009255063,
 -0.0055727884,
 -0.024418168,
 0.07281835,
 -0.0038550708,
 -0.03976652,
 0.0063

In [27]:
def retriever(query,top_k=3):
    query_embedding = ollama.embed(model=embedding_model, input=query)['embeddings'][0]
    
    similarities = []
    for chunk, embedding in VECTOR_DB:
        sim = cosine_similarity(query_embedding, embedding)
        similarities.append((chunk, sim))
    
    similarities.sort(key=lambda x: x[1], reverse=True)
    
    return [chunk for chunk, _ in similarities[:top_k]]

In [29]:
query = "which is first cat in space?"


In [30]:
results = retriever(query)
print("Top relevant facts:")
for i, fact in enumerate(results, 1):
    print(f"{i}. {fact.strip()}")

Top relevant facts:
1. The first cat in space was a French cat named Felicette (a.k.a. “Astrocat”) In 1963, France blasted the cat into outer space. Electrodes implanted in her brains sent neurological signals back to Earth. She survived the trip.
2. The first cat show was in 1871 at the Crystal Palace in London.
3. While it is commonly thought that the ancient Egyptians were the first to domesticate cats, the oldest known pet cat was recently found in a 9,500-year-old grave on the Mediterranean island of Cyprus. This grave predates early Egyptian art depicting cats by 4,000 years or more.
